## Step 2.1 — Install Packages

In [21]:
# Run once to install required packages
import sys
!{sys.executable} -m pip install -q --upgrade langchain-core langchain-google-genai google-generativeai google-genai python-dotenv langchain-text-splitters numpy

## Step 2.2 — Load Gemini API Key (Securely)

Instead of `getpass` (Colab approach), we load the key from a `.env` file.
The `.env` file is **never committed to git** (it's in `.gitignore`).

In [22]:
import os
from dotenv import load_dotenv

# Load key from .env file into environment
load_dotenv()

# LangChain's Gemini adapter looks for GOOGLE_API_KEY
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY", "")

if os.environ["GOOGLE_API_KEY"]:
    print("API key loaded into this session (not printed, not saved to the notebook file)")
else:
    print("ERROR: API key not found. Make sure .env file exists with GEMINI_API_KEY set.")

API key loaded into this session (not printed, not saved to the notebook file)


## Step 2.3 — Initialize the Gemini Model via LangChain

`ChatGoogleGenerativeAI` is a LangChain wrapper around Gemini's chat API.
Once wrapped, it behaves the same as any other LangChain chat model — this is the "swappable model" benefit.

In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",  # fast, low-cost Gemini model — good default for a workshop
)

print("Model initialized:", llm.model)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Model initialized: gemini-3.5-flash-lite


## Step 2.4 — Make Our First LLM Call

`llm.invoke()` sends a prompt to Gemini and returns an `AIMessage` object.
`response.content` holds the reply — but depending on the model it can be a plain string **or** a list of content blocks.
We write one helper once so the rest of the notebook always gets a clean string.

In [24]:
# First LLM call
response = llm.invoke("In one sentence, explain what a Large Language Model is.")

def get_text(response):
    """Return just the plain text of an LLM response, whether `.content` is a string
    or a list of content blocks (e.g. text + an internal thought-signature block)."""
    content = response.content
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )

print(get_text(response))

A Large Language Model is an advanced artificial intelligence system trained on vast amounts of text to understand, summarize, generate, and converse in human-like language.


## Section 3 — LLM Parameters

| Parameter | What it controls | Typical range |
|---|---|---|
| `temperature` | Randomness / creativity of output | 0.0 (deterministic) → 2.0 (more random) |
| `max_output_tokens` | Hard cap on response length | depends on model, e.g. 1–8192 |
| `model` | Which model answers — trades speed/cost vs capability | e.g. gemini-3.5-flash-lite vs gemini-1.5-pro |

### 3.1 Temperature: low vs. high, same prompt

- `temperature=0.0` → model always picks the most likely next token → nearly identical outputs each run
- `temperature=1.0` → model sometimes picks less-likely tokens → more varied, creative outputs

**When to use which:**
- Low temp → factual tasks, data extraction, classification, code generation
- High temp → brainstorming, marketing copy, creative writing

In [25]:
prompt = "Give me a one-sentence tagline for a coffee shop."

llm_low_temp  = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.0)
llm_high_temp = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=1.0)

print("=== temperature = 0.0 (run 3 times) ===")
for i in range(3):
    print(f"{i+1}.", get_text(llm_low_temp.invoke(prompt)))

print("\n=== temperature = 1.0 (run 3 times) ===")
for i in range(3):
    print(f"{i+1}.", get_text(llm_high_temp.invoke(prompt)))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


=== temperature = 0.0 (run 3 times) ===


c:\Users\Bacancy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


1. "Fueling your daily grind, one perfect cup at a time."


c:\Users\Bacancy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


...

1. "Fueling your daily grind, one perfect cup at a time."


c:\Users\Bacancy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


2. Fuel your hustle, one handcrafted cup at a time.


c:\Users\Bacancy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


3. Fuel your hustle, one perfectly roasted cup at a time.


### 3.2 Maximum Output Tokens

Tokens are roughly "chunks of text" (not words, not characters — somewhere in between).
`max_output_tokens` puts a hard ceiling on response length — useful for controlling cost, latency, or forcing short answers.

In [26]:
prompt = "Explain how a vector database works."

llm_short = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.3, max_output_tokens=20)
llm_long  = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.3, max_output_tokens=300)

print("=== max_output_tokens = 20 ===")
print(get_text(llm_short.invoke(prompt)))

print("\n=== max_output_tokens = 300 ===")
print(get_text(llm_long.invoke(prompt)))

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


=== max_output_tokens = 20 ===


c:\Users\Bacancy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


To understand how a **vector database** works, it helps to first understand what

=== max_output_tokens = 300 ===


c:\Users\Bacancy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


To understand how a **vector database** works, it helps to first understand what a "vector" is in the context of modern data, and why traditional databases struggle with it.

---

### 1. The Core Concept: What is a Vector?

Traditional databases (like MySQL, PostgreSQL, or MongoDB) store data as exact values: strings, numbers, and dates. They are great at answering questions like: *"Find all users named John who are over 30."*

However, modern AI deals with unstructured data—like text, images, audio, and video. To make this data searchable by computers, it is passed through a Machine Learning model (an **embedding model**) that converts it into a **vector** (an array of numbers, usually hundreds or thousands of dimensions long). 

Think of a vector as **coordinates on a giant, multi-dimensional map**. 
* **Words, sentences, or images that mean similar things are placed close to each other on this map.** 
* Concepts that are unrelated are placed far apart.

*Example:* 
* The vector for 

### 3.3 — Model Selection

Gemini offers several model sizes — smaller/faster/cheaper vs. larger/slower/more capable.
Rather than trusting a hardcoded name, let's ask the API what's actually available right now.

In [27]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Models available to your key that support chat:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(" -", m.name)

Models available to your key that support chat:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/gemini-3-pro-image
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-3.5-flash
 - models/gemini-3.5-flash-lite
 - models/gemini-omni-flash-preview
 - models/gemini-omni-1.1-flash
 - models/gemini-3.5-transcribe
 - models/gemini-3.6-flash
 - models/gemini-3.7-flash
 - m

In [28]:
import time

# Compare a lighter/faster model vs. a fuller model on the same prompt
prompt = "Explain the difference between Agentic AI and RAG in 2 sentences."

for model_name in ["gemini-3.5-flash", "gemini-3.1-flash-lite"]:
    fast_llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.3)
    start = time.time()
    result = fast_llm.invoke(prompt)
    elapsed = time.time() - start
    print(f"--- {model_name} ({elapsed:.1f}s) ---")
    print(get_text(result))
    print()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


--- gemini-3.5-flash (3.0s) ---
**RAG (Retrieval-Augmented Generation)** is a technique that enhances an AI's responses by fetching relevant, up-to-date information from an external database to answer specific queries accurately. In contrast, **Agentic AI** refers to autonomous systems that can proactively plan, use various tools, and execute multi-step workflows to achieve complex goals with minimal human intervention.

--- gemini-3.1-flash-lite (2.2s) ---
RAG (Retrieval-Augmented Generation) is a technique that provides an AI with external data to improve the accuracy of its responses to specific queries. Agentic AI goes further by enabling the system to autonomously plan, use tools, and take iterative actions to achieve complex, multi-step goals.



**What to notice:** `gemini-3.1-flash-lite` usually answers noticeably faster, while `gemini-3.5-flash` tends to give a more thorough/nuanced answer — that speed-vs-capability tradeoff is the main thing you're choosing when you pick a model.

In production you'd pick based on your actual requirements (latency budget, cost budget, task difficulty) — not just "use the best model available."

> If either model name above errors out for your key/region, re-run the previous cell and copy an exact name from the printed list.

## Section 4 — Build a Basic Chatbot

The simplest possible chatbot is just:

```
User → LLM → Response
```

No memory, no history — every message is a fresh, independent call.
Let's build that and immediately see its limitation.

In [29]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.7)

def chat(user_message):
    response = llm.invoke(user_message)
    return get_text(response)

print(chat("Hi! My name is Alex."))
print()
print(chat("What is my name?"))  # The model won't know — no history was sent

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Hi Alex! It’s nice to meet you. How are you doing today? Is there anything I can help you with?

I don’t know your name. As an AI, I don’t have access to your personal identity or private information unless you have shared it with me in this specific conversation.


**Expected behavior:** the model won't know your name on the second call — it will say it doesn't know, or ask you to tell it.

This is **not a bug**. `chat()` calls `llm.invoke(...)` fresh each time with *only* the new message. The previous exchange was never sent along, so as far as Gemini is concerned, the second call is a brand-new conversation.

**This is true of essentially all LLM APIs** — the model itself is stateless. Any "memory" you experience in products like Gemini or ChatGPT is the *application* resending the conversation history each time — which is exactly what we'll build next.

## Section 5 — Conversation History & Memory

To make a chatbot feel continuous, the application must keep track of what's been said and **resend it** on every call.

**Key terms:**

| Term | Definition |
|---|---|
| **Chat history** | The ordered list of messages exchanged so far (`Human: ...`, `AI: ...`). The raw transcript. |
| **Session** | One logical, continuous conversation instance (often identified by a session/conversation ID). A user may have many sessions over time. |
| **Short-term memory** | The (usually recent) chat history actively resent to the model for context in the *current* session. What we're building now. |
| **Long-term memory** | Information extracted and *persisted beyond a single session* (e.g. saved to a database or vector store) — "remembers you're vegetarian" three weeks later, not just three messages ago. |

We're building **short-term memory** — a simple in-memory list of messages, kept only for the life of this notebook session.

### 5.1 — LangChain Message Types

LangChain represents a conversation as a list of typed messages:

- `SystemMessage` — instructions for how the model should behave
- `HumanMessage` — something the user said
- `AIMessage` — something the model said

In [30]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Start with a system message that sets the model's behaviour for this session
history = [SystemMessage(content="You are a friendly, concise assistant.")]

def chat_with_history(user_message):
    history.append(HumanMessage(content=user_message))
    response = llm.invoke(history)        # send the WHOLE conversation so far
    reply_text = get_text(response)
    history.append(AIMessage(content=reply_text))
    return reply_text

print(chat_with_history("My name is Alex."))

Hi Alex! It's nice to meet you. How can I help you today?


**Expected behavior:** this time the model correctly answers "Alex" — because the full `history` list (system prompt + every human/AI turn so far) is sent to the model on *every single call*.

Nothing is stored on Google's servers between calls — **we** are resending the transcript each time, and the model re-reads it from scratch.

Run the cell below to see exactly what's being sent under the hood:

In [31]:
# Inspect what's actually being sent to the model on every call
for msg in history:
    print(f"[{msg.type}] {msg.content}")

[system] You are a friendly, concise assistant.
[human] My name is Alex.
[ai] Hi Alex! It's nice to meet you. How can I help you today?


**A practical consequence:** the longer a conversation gets, the more text you resend (and pay for/wait on) with every single turn. This is why real chat products eventually summarize or trim older history instead of keeping it forever. We won't build that today, but it's worth knowing it's the next problem you'd hit.

**Recap of the four memory terms:**

| Term | What it is | Lifespan |
|---|---|---|
| **Chat history** | The raw list of messages | As long as you keep the list |
| **Session** | The conversation instance the history belongs to | Usually one user visit/interaction |
| **Short-term memory** | History actively resent to the model for context | Current session only |
| **Long-term memory** | Facts deliberately saved for future sessions | Persists across sessions (needs a DB/vector store) |

## Section 6 — Introduce RAG (Retrieval-Augmented Generation)

**Why not just paste the whole document into the prompt?**
- LLMs have a limited **context window** — very large or many documents simply won't fit.
- Even when it fits, sending a huge document on *every* question is slow and expensive (you pay/wait per token, every single call).
- Irrelevant surrounding text can distract the model and increase hallucination risk — needle-in-a-haystack problems are real.
- Real knowledge bases (wikis, ticket histories, codebases) are usually far bigger than any context window anyway.

**The idea behind RAG:** instead of sending the whole document, *search it first* for the few pieces actually relevant to the current question, and send only those pieces to the LLM.

```
Document
   ↓ Load
   ↓ Chunk
   ↓ Embedding
   ↓ Vector Database
   ↓ Similarity Search
   ↓ Relevant Chunks
   ↓ LLM
   ↓ Answer
```

**What each step means:**

| Step | What happens |
|---|---|
| **Load** | Read the raw source (PDF, wiki, text file) into memory |
| **Chunk** | Split into smaller, overlapping pieces — each about one specific topic |
| **Embedding** | Convert each chunk into a vector (list of numbers) capturing its *meaning* |
| **Vector database** | Store optimized for holding vectors and finding the closest ones fast |
| **Similarity search** | Embed the user's question, find chunks with the closest vectors (most semantically relevant) |
| **Relevant chunks** | The small handful that actually matter for this question (out of possibly thousands) |
| **LLM** | Given the question + those chunks as context, generate a grounded answer |
| **Answer** | Final response, ideally using only facts from the retrieved chunks |

Let's build this end-to-end with a tiny sample document.

## Section 7 — Build a Small RAG System

To keep things self-contained, we'll use a small fictional company handbook as our "document" — no file upload needed.
It contains specific facts we can later ask questions about, and intentionally leaves some things unanswered.

In [32]:
sample_document = """
Acme Robotics — Employee Handbook (Excerpt)

Remote Work Policy:
Acme Robotics operates on a hybrid model. Employees are expected to work from the office
on Tuesdays and Thursdays. All other weekdays may be worked remotely, subject to manager
approval. Fully remote arrangements require VP-level sign-off and are reviewed quarterly.

Leave Policy:
Full-time employees accrue 18 days of paid vacation per year, plus 10 paid sick days.
Vacation requests must be submitted at least 5 business days in advance through the HR
portal. Unused vacation days may be carried over, up to a maximum of 5 days, into the
next calendar year.

Expense Reimbursement:
Employees may claim reimbursement for approved business expenses (travel, client meals,
conference fees) by submitting receipts within 30 days of the expense. Reimbursements
are processed within 10 business days of approval. Personal expenses, including gym
memberships and home internet, are not reimbursable.

Equipment Policy:
New employees receive a laptop and a one-time $200 home-office setup stipend during
their first 90 days. Equipment remains the property of Acme Robotics and must be
returned upon termination of employment.
"""

print(sample_document[:200], "...")


Acme Robotics — Employee Handbook (Excerpt)

Remote Work Policy:
Acme Robotics operates on a hybrid model. Employees are expected to work from the office
on Tuesdays and Thursdays. All other weekdays ...


### 7.1 — Document Loader

In a real project you'd use a loader like `PyPDFLoader` or `TextLoader` to read from disk.
Since our text already lives in the notebook, we wrap it in LangChain's `Document` object — the same shape every loader produces.

In [33]:
from langchain_core.documents import Document

docs = [Document(page_content=sample_document, metadata={"source": "acme_handbook.txt"})]
print(f"Loaded {len(docs)} document(s), {len(docs[0].page_content)} characters.")

Loaded 1 document(s), 1188 characters.


### 7.2 — Text Splitter (Chunking)

`RecursiveCharacterTextSplitter` splits on natural boundaries (paragraphs → sentences → words) so chunks stay coherent.
`chunk_overlap` repeats a little text between consecutive chunks so context isn't lost at a boundary.

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
chunks = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks.\n")
for i, c in enumerate(chunks):
    print(f"--- Chunk {i+1} ---\n{c.page_content}\n")

Split into 6 chunks.

--- Chunk 1 ---
Acme Robotics — Employee Handbook (Excerpt)

--- Chunk 2 ---
Remote Work Policy:
Acme Robotics operates on a hybrid model. Employees are expected to work from the office
on Tuesdays and Thursdays. All other weekdays may be worked remotely, subject to manager
approval. Fully remote arrangements require VP-level sign-off and are reviewed quarterly.

--- Chunk 3 ---
Leave Policy:
Full-time employees accrue 18 days of paid vacation per year, plus 10 paid sick days.
Vacation requests must be submitted at least 5 business days in advance through the HR
portal. Unused vacation days may be carried over, up to a maximum of 5 days, into the
next calendar year.

--- Chunk 4 ---
Expense Reimbursement:
Employees may claim reimbursement for approved business expenses (travel, client meals,
conference fees) by submitting receipts within 30 days of the expense. Reimbursements
are processed within 10 business days of approval. Personal expenses, including gym

--- 

### 7.3 — Embeddings

Turn each chunk into a vector using Gemini's embedding model.
We auto-detect the first available embedding model from the API so the name stays valid regardless of region.

In [35]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Use gemini-embedding-001 — Google's dedicated text embedding model
embedding_model_name = "models/gemini-embedding-001"
print("Using embedding model:", embedding_model_name)

embeddings = GoogleGenerativeAIEmbeddings(model=embedding_model_name)

# Sanity check: embed one chunk and inspect the vector
sample_vector = embeddings.embed_query(chunks[1].page_content)
print(f"Embedding vector length: {len(sample_vector)}")
print("First 8 numbers:", sample_vector[:8])

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Using embedding model: models/gemini-embedding-001
Embedding vector length: 3072
First 8 numbers: [0.00014219982, 0.031407684, 0.0021138007, -0.050392672, -0.013205063, 0.007194772, 0.034247287, -0.013564435]


**What you're looking at:** those ~3000 numbers represent the *meaning* of a chunk. They mean nothing to a human, but two chunks about similar topics will have vectors that are mathematically close together — that's the entire trick similarity search relies on.

### 7.4 — Vector Store

`InMemoryVectorStore` keeps everything in RAM — zero infrastructure, perfect for a workshop.
In production you'd use something persistent like Chroma, FAISS, Pinecone, or pgvector.

In [36]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(chunks)

print(f"Vector store now holds {len(chunks)} embedded chunks.")

Vector store now holds 6 embedded chunks.


### 7.5 — Retriever + Similarity Search

A **retriever** wraps "embed the query → search the vector store" into one call.
Let's test it directly before wiring it to the LLM, so we can see exactly what gets retrieved.

In [37]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})  # return top-2 most relevant chunks

results = retriever.invoke("How much vacation time do I get?")
for i, doc in enumerate(results):
    print(f"--- Retrieved chunk {i+1} ---\n{doc.page_content}\n")

--- Retrieved chunk 1 ---
Leave Policy:
Full-time employees accrue 18 days of paid vacation per year, plus 10 paid sick days.
Vacation requests must be submitted at least 5 business days in advance through the HR
portal. Unused vacation days may be carried over, up to a maximum of 5 days, into the
next calendar year.

--- Retrieved chunk 2 ---
Equipment Policy:
New employees receive a laptop and a one-time $200 home-office setup stipend during
their first 90 days. Equipment remains the property of Acme Robotics and must be
returned upon termination of employment.



**Expected behavior:** the leave-policy chunk should come back as the top match, even though the question uses "vacation time" and the document uses "paid vacation" — the embedding is capturing *meaning*, not just keyword matching.

### 7.6 — Wire it up to Gemini (Q&A function)

The most important line in a RAG system: **tell the model to answer only from the provided context**.
Without this instruction, the model will happily guess using its training data.

In [38]:
RAG_PROMPT = """Answer the question using ONLY the context below.
If the answer is not contained in the context, respond exactly with:
"I don't have that information in the document."
Do not use outside knowledge and do not guess.

Context:
{context}

Question: {question}

Answer:"""

def ask(question):
    relevant_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in relevant_docs)
    prompt = RAG_PROMPT.format(context=context, question=question)
    return get_text(llm.invoke(prompt))

# Question whose answer IS in the document
print(ask("How many vacation days do full-time employees get?"))
print()
# Question whose answer is NOT in the document
print(ask("Does Acme Robotics offer a 401k match?"))

Full-time employees accrue 18 days of paid vacation per year.

I don't have that information in the document.


**Expected behavior:**
- First `ask()` → correctly answers "18 days" from the Leave Policy chunk.
- Second `ask()` (401k) → `"I don't have that information in the document."` — even though the model *could* answer from general knowledge, we told it not to use outside knowledge.

That's the difference between a RAG system and "just asking an LLM."

---

## Section 8 — Final Mini AI System: Putting It All Together

Combining everything from today into one small application:

```
User
 ↓ Chat Interface
 ↓ Conversation History   (Section 5)
 ↓ Retriever              (Section 7)
 ↓ Relevant Context       (Section 7)
 ↓ Gemini                 (Section 2/3)
 ↓ Response
```

On every turn we:
1. Retrieve context relevant to the *new* question
2. Combine it with the conversation history so far
3. Send everything to Gemini
4. Record the new turn into history for next time

In [39]:
conversation_history = []  # list of HumanMessage / AIMessage, growing each turn

def rag_chat(user_message):
    # 1. Retrieve document context relevant to THIS message
    relevant_docs = retriever.invoke(user_message)
    context = "\n\n".join(doc.page_content for doc in relevant_docs)

    # 2. Build fresh instructions (including retrieved context) for this turn
    system_message = SystemMessage(content=(
        "You are a helpful assistant for Acme Robotics employees. "
        "Use the CONTEXT below if it's relevant to the question. "
        "If the answer isn't in the context AND isn't something already established "
        "earlier in this conversation, say you don't have that information. "
        "Do not make up policy details.\n\nCONTEXT:\n" + context
    ))

    # 3. Combine: system instructions + everything said so far + the new message
    messages = [system_message] + conversation_history + [HumanMessage(content=user_message)]
    response = llm.invoke(messages)
    reply_text = get_text(response)

    # 4. Record this turn so future turns remember it
    conversation_history.append(HumanMessage(content=user_message))
    conversation_history.append(AIMessage(content=reply_text))

    return reply_text

print(rag_chat("Hi, my name is Alex."))
print()
print(rag_chat("How many sick days do I get per year?"))
print()
print(rag_chat("And what's my name again?"))

Hello, Alex! How can I help you with Acme Robotics policies today?

Full-time employees at Acme Robotics receive 10 paid sick days per year.

Your name is Alex.


**Expected behavior:**
- Turn 1 — friendly greeting; memory now records "user's name is Alex"
- Turn 2 — sick-day fact pulled from the retrieved handbook chunk (RAG working)
- Turn 3 — answered from `conversation_history`, not the document (memory working)

Both memory and retrieval cooperating in the same system.

### Optional: Live Interactive Demo

Type messages below, type `quit` to stop. Try asking things that are in the handbook, things that aren't, and things about earlier in your own conversation.

In [40]:
# Interactive demo — run this cell, type messages, type "quit" to stop.
while True:
    user_input = input("You: ")
    if not user_input.strip():          # skip empty / Escape
        continue
    if user_input.strip().lower() == "quit":
        break
    print("AI:", rag_chat(user_input))

AI: Hello! How can I assist you with Acme Robotics policies today?
AI: Your name is Alex.


---

## Recap: What You Built Today

```
LLM → Parameters → Chatbot → History → Memory → Embeddings → Vector DB → Retrieval → RAG → AI Application
```

You started with a single stateless call to Gemini and ended with an application that:

- **Remembers** the conversation so far (short-term memory)
- **Looks up facts** from a real document instead of relying on the model's own knowledge (RAG)
- **Refuses to answer** when it genuinely doesn't know (grounded, not hallucinating)
- **Combines** all of that into one coherent multi-turn experience

That's already meaningfully more than "call an LLM API" — it's a small, real **AI system**, with the same shape that production AI products use, just at a smaller scale.

---

### Natural Next Steps

| Topic | What it unlocks |
|---|---|
| **Tools / function calling & agents** | LLM can call your own code/APIs to take actions, not just answer questions |
| **Persistent long-term memory** | Key facts survive across sessions (saved to a real DB) |
| **Production-grade vector stores** | Chroma, FAISS, pgvector, or managed services instead of in-memory |
| **Evaluation & guardrails** | Systematically test accuracy, safety, and groundedness before shipping |
| **Streaming responses** | Send tokens to the user as they're generated instead of waiting for the full answer |